In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import sklearn as sk
import requests

# Configurações de display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Configurar endpoint da API do BACEN
import sgs
sgs.api = 'https://api.bcb.gov.br/dados/serie/bcdata.sgs.11/dados'

## FK Partners Fundamentos de Derivativos

### Tópicos abordados: Derivativos, Contratos futuros, Contratos a termo, Swap, Opções e Operações com Opções. Para fins didáticos, não serão considerados os spreads das operações.

 ##### Derivativos
É um instrumento financeiro, cujo preço deriva de um ativo ou instrumento financeiro de referência (ativo-objeto), isto é, sua formação de preço depende de outro ativo. É um mercado de liquidação futura. Podem ser divididos em: Futuros/Termos, Opção e Swap.

Hedge: deseja se proteger contra o risco de mercado

Especulação: busca lucrar com a variação de preços, assumindo o risco de mercado (o clássico comprar na baixa e vender na alta).

Arbitragem: busca lucrar com a diferença de preços entre mercados, sem assumir risco de mercado 
(Exemplo: compra barato e imediatamente vende mais caro, quando há descasamento entre preços em bolsas diferentes).

Posicionamento: busca se posicionar em um mercado, sem necessariamente ter uma motivação clara de hedge, especulação ou arbitragem 
(exemplo: um investidor que compra ações de uma empresa porque acredita que ela é segura, sem necessariamente ter uma estratégia clara de investimento).

Alavancagem: busca amplificar os ganhos (ou perdas) de uma posição, utilizando instrumentos financeiros ou capital emprestado. 
(Exemplo: um investidor que compra opções para obter exposição a um ativo com um investimento inicial menor do que o necessário para comprar o ativo diretamente).

#1Q: Posicionamento: estou em dúvida se meu entendimento foi totalmente correto. 

#1A:

#2Q: Alavancagem: realmente não entendi muito bem.

#2A: 
 

##### Futuros/Swap

1.1 Definição:
Acordo entre vendedor e comprador para entregar e receber determinada quantidade de um ativo  por determinado preço para liquidação em uma data futura.
Principais componentes: quantidade, ativo, preço e data futura.
Exemplo: um produtor de soja não sabe por qual preço poderá vender seu ativo ao final da colheita, logo, pode travar a venda a R$10/unidade, com a promessa de entrega em jan/2027.
OBS: nos contratos a termo, existe a obrigação de cumprimento da operação, diferentemente do caso das Opções.

In [8]:
PRECO_ATUAL = 18
PRECO_FUTURO = 18
ATIVO = 'SOJA'
DATA_ATUAL = '2026-01-01'
DATA_FUTURA = '2027-01-01'

def calcular_resultado_operacao_futuro():
    valor_variacao = (PRECO_FUTURO - PRECO_ATUAL) / PRECO_ATUAL
    if PRECO_ATUAL > PRECO_FUTURO:
        print(f"""O preço de {ATIVO} diminuiu em {valor_variacao:.2%}, portanto, travar o preço foi positivo.\n  

""")

    elif PRECO_ATUAL < PRECO_FUTURO:
        print(f"""O preço de {ATIVO} aumentou em {valor_variacao:.2%}, portanto, travar o preço não foi necessariamente negativo,\n pois o produtor não perdeu, apenas deixou de ganhar. Além disso, pôde contar com a previsibilidade de sua operação.  

""")

    else:
        print(f"""O preço de {ATIVO} permaneceu inalterado, portanto, travar o preço não foi necessariamente negativo,\n pois o produtor não perdeu, apenas deixou de ganhar.\n Além disso, pôde contar com a previsibilidade de sua operação.  

""")

operacao_futuro = calcular_resultado_operacao_futuro()
print(operacao_futuro)          


O preço de SOJA permaneceu inalterado, portanto, travar o preço não foi necessariamente negativo,
 pois o produtor não perdeu, apenas deixou de ganhar.
 Além disso, pôde contar com a previsibilidade de sua operação.  


None


##### Opções

1.2 Definição:
Acordo entre vendedor e comprador, com a possibiliade de exercício do comprador e a obrigação de exercício do vendedor.
Similar a um seguro, a opção é uma operação de compra (como se fosse o pagamento de um prêmio) do direito de exercer ou não uma determinada operação no futuro, de acordo com um determinado cenário.
Exemplo: Deseja-se adquirir um ativo de PETR4 no futuro, o comprador pode comprar o direito de adquirir esse ativo a um determinado preço mais baixo (opção de compra), assim como pode adquirir a opção de venda futura a um determinado preço.

Outras nomenclaturas:
CALL: é uma opção de compra.
PUT: é uma opção de venda.

In [25]:
# Configurações para a simulação de opções randômicas
PRECO_ATUAL = np.random.randint(1, 11)
print(f"Preço atual do ativo: {PRECO_ATUAL}")
PRECO_FUTURO = np.random.randint(1, 11)
print(f"Preço futuro do ativo: {PRECO_FUTURO}")
# STRIKE = np.random.randint(1, 11)
# print(f"Preço de exercício (strike): {STRIKE}")
TIPO_OPCAO = np.random.choice(['CALL', 'PUT'])
print(f"Tipo de opção: {TIPO_OPCAO}")

ATIVO = 'PETR4'
DATA_ATUAL = '2026-01-01'
DATA_FUTURA = '2027-01-01'


def calcular_resultado_operacao_opcao():
    valor_variacao = (PRECO_FUTURO - PRECO_ATUAL) / PRECO_ATUAL
    if (PRECO_ATUAL > PRECO_FUTURO and TIPO_OPCAO == 'CALL'):
        print(f"""O preço de {ATIVO} diminuiu em {valor_variacao:.2%}, portanto, comprar uma CALL foi negativo, pois o operador comprou a opção de comprar mais barato um ativo que naturalmente se tornou mais barato.\n
              Nesse caso, o operador não fará exercício da opção, sofrendo o prejuízo do prêmio pago pela opção.\n""")
    elif (PRECO_ATUAL < PRECO_FUTURO and TIPO_OPCAO == 'CALL'):
        print(f"""O preço de {ATIVO} aumentou em {valor_variacao:.2%}, portanto, comprar uma CALL foi positivo, pois o operador comprou a opção de comprar mais barato um ativo que naturalmente se tornou mais caro.\n""")
    elif (PRECO_ATUAL > PRECO_FUTURO and TIPO_OPCAO == 'PUT'):
        print(f"""O preço de {ATIVO} diminuiu em {valor_variacao:.2%}, portanto, comprar uma PUT foi positivo, pois o operador comprou a opção de vender mais caro um ativo que naturalmente se tornou mais barato.\n""")
    elif (PRECO_ATUAL < PRECO_FUTURO and TIPO_OPCAO == 'PUT'):
        print(f"""O preço de {ATIVO} aumentou em {valor_variacao:.2%}, portanto, comprar uma PUT foi negativo, pois o operador comprou a opção de vender mais caro um ativo que naturalmente se tornou mais caro.\n Nesse caso, o operador não fará exeercício da opção, sofrendo o prejuízo do prêmio pago pela opção.\n""")
    else:
        print(f"""O preço de {ATIVO} permaneceu inalterado, portanto, comprar uma {TIPO_OPCAO} não foi necessariamente negativo de modo geral,\n apenas em relação ao pagamento do prêmio.\n""")


Preço atual do ativo: 4
Preço futuro do ativo: 5
Tipo de opção: PUT


In [26]:
operacao_opcao = calcular_resultado_operacao_opcao()

O preço de PETR4 aumentou em 25.00%, portanto, comprar uma PUT foi negativo, pois o operador comprou a opção de vender mais caro um ativo que naturalmente se tornou mais caro.
 Nesse caso, o operador não fará exeercício da opção, sofrendo o prejuízo do prêmio pago pela opção.



#### Swap

1.3 Definição: Acordo entre duas partes para troca de caixa no futuro. O acordo deve conter os indexadores,
datas de pagamento e cálculos que serão utilizados.
Exemplo: Uma contraparte tem um fluxo de pagamento indexado em IPCA e quer trocar para Selic, ela poderá
fazer um contrato de Swap para que seu pagamento seja em Selic e outra contraparte assuma o pagamento em IPCA. 

In [33]:
# Configurações para a simulação (bastante simplificada) de um Swap
TAXA_IPCA = np.random.uniform(0.02, 0.08)
TAXA_SELIC = np.random.uniform(0.02, 0.08)
print(f"Taxa IPCA (cenário futuro): {TAXA_IPCA:.2%}")
print(f"Taxa Selic (cenário futuro): {TAXA_SELIC:.2%}")

NOTIONAL = 100000

CONTRAPARTE_A = 'Banco Itaú'
CONTRAPARTE_B = 'Banco Monique'
DATA_ATUAL = '2026-01-01'
DATA_FUTURA = '2027-01-01'

# Indexadores de cada contraparte
INDEXADOR_A_ORIGINAL = 'IPCA'
INDEXADOR_B_ORIGINAL = 'Selic'


def calcular_operacao_swap():
    """
    Simula uma operação de Swap de taxa de juros (IPCA x Selic).
    
    Cada contraparte faz um pagamento em um indexador diferente.
    Sem o Swap: Contraparte A pagaria em IPCA, Contraparte B em Selic.
    Com o Swap: Contraparte A passa a receber em Selic e pagar em IPCA, Contraparte B passa a receber em IPCA e pagar em Selic.
    """
    
    # Cálculos dos fluxos de caixa
    # Cenário 1: SEM Swap (cada um mantém seu indexador)
    pagamento_A_sem_swap = NOTIONAL * TAXA_IPCA  # A paga em IPCA
    pagamento_B_sem_swap = NOTIONAL * TAXA_SELIC  # B paga em Selic
    
    # Cenário 2: COM Swap (trocam os indexadores)
    pagamento_A_com_swap = NOTIONAL * TAXA_SELIC  # A agora paga em Selic
    pagamento_B_com_swap = NOTIONAL * TAXA_IPCA   # B agora paga em IPCA
    
    # Análise de ganho/perda para cada contraparte
    diferenca_A = pagamento_A_sem_swap - pagamento_A_com_swap
    diferenca_B = pagamento_B_sem_swap - pagamento_B_com_swap
    
    print(f"SIMULAÇÃO DE SWAP - {CONTRAPARTE_A} x {CONTRAPARTE_B}")
    print(f"\nPeríodo: {DATA_ATUAL} a {DATA_FUTURA}")
    print(f"Notional: R$ {NOTIONAL:,.2f}")
    print(f"\n{'CENÁRIO SEM SWAP (Estrutura Atual)':-^70}")
    print(f"{CONTRAPARTE_A} paga em {INDEXADOR_A_ORIGINAL:8} → R$ {pagamento_A_sem_swap:,.2f}")
    print(f"{CONTRAPARTE_B} paga em {INDEXADOR_B_ORIGINAL:8} → R$ {pagamento_B_sem_swap:,.2f}")
    
    print(f"\n{'CENÁRIO COM SWAP (Após Contrato)':-^70}")
    print(f"{CONTRAPARTE_A} paga em {INDEXADOR_B_ORIGINAL:8} → R$ {pagamento_A_com_swap:,.2f}")
    print(f"{CONTRAPARTE_B} paga em {INDEXADOR_A_ORIGINAL:8} → R$ {pagamento_B_com_swap:,.2f}")
    
    print(f"\n{'ANÁLISE DE RESULTADO':-^70}")
    if diferenca_A > 0:
        print(f"\n {CONTRAPARTE_A} se BENEFICIA do Swap:")
        print(f"  Economia: R$ {diferenca_A:,.2f}")
        print(f"  Pagava {TAXA_IPCA:.2%} em {INDEXADOR_A_ORIGINAL}, agora paga {TAXA_SELIC:.2%} em {INDEXADOR_B_ORIGINAL}")
    elif diferenca_A < 0:
        print(f"\n {CONTRAPARTE_A} PERDE com o Swap:")
        print(f"  Custo adicional: R$ {abs(diferenca_A):,.2f}")
        print(f"  Pagava {TAXA_IPCA:.2%} em {INDEXADOR_A_ORIGINAL}, agora paga {TAXA_SELIC:.2%} em {INDEXADOR_B_ORIGINAL}")
    else:
        print(f"\n {CONTRAPARTE_A} fica INDIFERENTE com o Swap (cenário neutro)")
    
    if diferenca_B > 0:
        print(f"\n {CONTRAPARTE_B} se BENEFICIA do Swap:")
        print(f"  Economia: R$ {diferenca_B:,.2f}")
        print(f"  Pagava {TAXA_SELIC:.2%} em {INDEXADOR_B_ORIGINAL}, agora paga {TAXA_IPCA:.2%} em {INDEXADOR_A_ORIGINAL}")
    elif diferenca_B < 0:
        print(f"\n  {CONTRAPARTE_B} PERDE com o Swap:")
        print(f"  Custo adicional: R$ {abs(diferenca_B):,.2f}")
        print(f"  Pagava {TAXA_SELIC:.2%} em {INDEXADOR_B_ORIGINAL}, agora paga {TAXA_IPCA:.2%} em {INDEXADOR_A_ORIGINAL}")
    else:
        print(f"\n {CONTRAPARTE_B} fica INDIFERENTE com o Swap (cenário neutro)")




Taxa IPCA (cenário futuro): 5.66%
Taxa Selic (cenário futuro): 5.53%


In [34]:
operacao_swap = calcular_operacao_swap()

SIMULAÇÃO DE SWAP - Banco Itaú x Banco Monique

Período: 2026-01-01 a 2027-01-01
Notional: R$ 100,000.00

------------------CENÁRIO SEM SWAP (Estrutura Atual)------------------
Banco Itaú paga em IPCA     → R$ 5,663.55
Banco Monique paga em Selic    → R$ 5,527.80

-------------------CENÁRIO COM SWAP (Após Contrato)-------------------
Banco Itaú paga em Selic    → R$ 5,527.80
Banco Monique paga em IPCA     → R$ 5,663.55

-------------------------ANÁLISE DE RESULTADO-------------------------

 Banco Itaú se BENEFICIA do Swap:
  Economia: R$ 135.75
  Pagava 5.66% em IPCA, agora paga 5.53% em Selic

  Banco Monique PERDE com o Swap:
  Custo adicional: R$ 135.75
  Pagava 5.53% em Selic, agora paga 5.66% em IPCA


In [35]:
### Perguntas:
#1 Não deveriam se zerar? Nessa simulação, é como se alguém sempre saísse no prejuízo
#2 Exemplo, se a contraparte escolhe trocar seu IPCA pra pré-fixada, mesmo sabendo que pode pagar mais caro, mas tem a previsibilidade de saber exatamente quanto vai pagar, 
# isso não é tão benéfico assim, certo?



#### 2.1 Contratos Futuros

- Diferença entre contratos à vista e futuros: em operações spot, o operador dispende do valor integral da aquisição do ativo,
- Nos contratos futuros, o operador dispende de um valor menor no presente com a promessa de aquisição futura daquele ativo (alavancagem).
- As operações de contratos futuros são feitos sempre por intermédio da B3 (somente balcão, assim como Swap). Ou seja, uma contraparte compra/vende contra a bolsa um determinado contrato e outra contraparte compra/vende contra a bolsa esse mesmo contrato.
- Existe sempre a obrigatoriedade entre as partes, ou seja, ao final da liquidação da operação, o comprador deve exercer a compra e o vendedor deve exercer a venda.
- Não existe movimentação financeira no início das operações.


#1Q: Como funciona o "mercado secundário" dessas operações? Ou seja, até o vencimento, posso ir comprando e vendendo N vezes?: 

#1A: Sim, pode comprar e vender N vezes o mesmo contrato. Os contratos da bolsa são padronizados, ou seja, todo contrato de ativo X, com tamanho y e vencimento
em 01/01/2027 pertencem ao mesmo contrato, então pode-se vender e comprar esse contrato durante sua vigência e, inclusive, ir lucrando desta forma até o final da operação.

#2Q: Por que necessariamente precisa do intermédio da Bolsa?

#2A:

#Q3: Ou seja, estar acima da linha é sempre positivo então, certo?

#3A:


In [20]:
# Faça, usando plotly, dois gráficos: um de posição de venda de NDF e outro de compra de NDF, sendo o eixo Y o resultado da operação (ganho ou perda) e o eixo X o preço futuro do ativo. Considere um preço spot 15 e preço vencimento 20.
# Configurações para os gráficos de NDF
PRECO_SPOT_NDF_1 = 15 # Cenário positivo para compra (LONG), negativo para venda (SHORT)
PRECO_SPOT_NDF_2 = 25 # Cenário positivo para venda (SHORT), negativo para compra (LONG)
PRECO_VENCIMENTO_NDF = 20
NOTIONAL_NDF = 100000

# Array de preços futuros
precos_futuros = np.linspace(10, 30)

# Cálculo de resultados
resultado_ndf_compra = (precos_futuros - PRECO_VENCIMENTO_NDF) * NOTIONAL_NDF
resultado_ndf_venda = (PRECO_VENCIMENTO_NDF - precos_futuros) * NOTIONAL_NDF

# Dataframe
df_ndf = pd.DataFrame({
    'Preço Futuro': precos_futuros,
    'Resultado Compra': resultado_ndf_compra,
    'Resultado Venda': resultado_ndf_venda
})

# Gráfico 1: Compra (LONG) - Cenário 1: Comprado em 15, Vencimento em 20
fig_compra = px.line(df_ndf, x='Preço Futuro', y='Resultado Compra', 
                      title='NDF - Posição LONG (Compra)')
fig_compra.update_traces(line=dict(color='green', width=3))
fig_compra.add_vline(x=PRECO_VENCIMENTO_NDF, line_dash="solid", line_color="gray", opacity=0.7, 
                     annotation_text="Vencimento: 20", annotation_position="top right")
fig_compra.add_scatter(x=[PRECO_SPOT_NDF_1], y=[0], mode='markers', 
                       marker=dict(color='blue', size=10), showlegend=False,
                       hovertemplate=f"Spot 1: {PRECO_SPOT_NDF_1}")
fig_compra.add_scatter(x=[PRECO_SPOT_NDF_2], y=[0], mode='markers', 
                       marker=dict(color='yellow', size=10, line=dict(color='black', width=1)), showlegend=False,
                       hovertemplate=f"Spot 2: {PRECO_SPOT_NDF_2}")
fig_compra.update_layout(
    hovermode='x unified', 
    template='plotly_white', 
    width=500, 
    height=300,
    yaxis=dict(showticklabels=False, title='Ganho/Perda')
)
fig_compra.show()

# Gráfico 2: Venda (SHORT) - Cenário 2: Comprado em 25, Vencimento em 20
fig_venda = px.line(df_ndf, x='Preço Futuro', y='Resultado Venda', 
                     title='NDF - Posição SHORT (Venda)')
fig_venda.update_traces(line=dict(color='green', width=3))
fig_venda.add_vline(x=PRECO_VENCIMENTO_NDF, line_dash="solid", line_color="gray", opacity=0.7,
                    annotation_text="Vencimento: 20", annotation_position="top right")
fig_venda.add_scatter(x=[PRECO_SPOT_NDF_1], y=[0], mode='markers', 
                      marker=dict(color='blue', size=10), showlegend=False,
                      hovertemplate=f"Spot 1: {PRECO_SPOT_NDF_1}")
fig_venda.add_scatter(x=[PRECO_SPOT_NDF_2], y=[0], mode='markers', 
                      marker=dict(color='yellow', size=10, line=dict(color='black', width=1)), showlegend=False,
                      hovertemplate=f"Spot 2: {PRECO_SPOT_NDF_2}")
fig_venda.update_layout(
    hovermode='x unified', 
    template='plotly_white', 
    width=500, 
    height=300,
    yaxis=dict(showticklabels=False, title='Ganho/Perda')
)
fig_venda.show()

#### 2.2 Precificação de um contrato futuro

Preço spot (S): preço à vista.
Preço futuro (F): preço à vista acrescido da taxa de juros (i) livre de risco.
Taxa de juros livre de risco (i):  
Tempo (T): Duração da operação.
Base: diferença entre o preço futuro e preço à vista.

F = S * (1 + i)^T

In [25]:
S = 1000
i = 0.05
t = 10
F = S * (1 + i) ** t
base = F - S

print(f"Valor spot: {S}")
print(f"Valor futuro: {F}")
print(f"Diferença do valor spot: {base}")

Valor spot: 1000
Valor futuro: 1628.894626777442
Diferença do valor spot: 628.894626777442


Contratos de commodities: existe uma singularidade para este tipo de contrato que é o custo de manter essa opeeração, que pode ser dividida em:
- Custo de Carrego (CC): relativo a custos de frete e estocagem (entre outros), expresso em unidades monetárias.
- Rendimento de conveniência: corresponde ao benefício de manter a mercadoria em estoque a fim de assegurar o atendimento de uma demanda futura, expresso como taxa percentual.

A fórmula rearanjada fica:

F = S × [(1 + i - Y) / (1 + T)] 
ou
 F × (1 + Y)^T = S × (1 + i)^T + CC

Por que CC "vira" Y:

O custo de carrego (CC) é expresso em valores absolutos ($), mas para comparar com a taxa de juros (i), ele precisa ser convertido em taxa percentual. O rendimento de conveniência (Y) é essa conversão:

Y = CC / S (custo como percentual do preço spot)

Dessa forma, o rendimento de conveniência reduz a taxa de juros efetiva da operação, porque mantendo a commodity em estoque, você economiza custos futuros.

Interpretação:
- Se não houvesse custo de carrego nem rendimento de conveniência: F = S × (1 + i)^T (fórmula simples)
- Com conveniência: F = S × (1 + i - Y)^T (a taxa efetiva diminui porque há benefício em manter o estoque)
F * (1 + Y) **T = S * (1 + i) **T + CC

#### Operações na B3 - Definições importantes

Margem de garantia referente a volatividade: para operar na Bolsa, o investidor deve depositar margem de garantia a fim de cobrir suas obrigações.
Imagine que você quer comprar um contrato futuro de soja. Em vez de pagar o valor total hoje, você deposita apenas uma "garantia" (como um depósito caução). Por que? Porque o contrato será acertado dia a dia. Se o preço cair demais, você pode perder dinheiro rapidinho, então a bolsa pede esse depósito para garantir que você terá recursos para cumprir suas obrigações. É como deixar uma caução para alugar um imóvel.
#1Q: Ou seja, existe sim um pagamento à vista pra operar contratos futuros? Achei que isso ocorria somente pra Opção.

#1A: 

Contraparte central: a B3 assume as posições de comprador perante o vendedor e vice-versa.
#2Q: Então a B3 cobra alguma taxa a mais para operar como uma clearing house ou somente a margem de garantia?

#2A: 

Posição em aberto: posição líquida fixada para um único vencimento de um único contrato. Exemplo: você entra comprado em 10 contratos. No dia seguinte, vende 3. Posição em aberto: 7 contratos comprados aguardando seu desfecho.

Ajustes diários (Mark-to-Market): posições mantidas em aberto são acertadas todos os dias (elimina o risco de crédito), segundo o preço de ajuste do dia. Todo dia, a bolsa recalcula quanto sua posição vale ao preço de fechamento. Se você está comprado e o preço subiu, você ganha dinheiro (entra na sua conta). Se o preço caiu, você perde (sai da sua conta). Isso acontece todos os dias, não só no vencimento. Exemplo:
Dia 1: preço sobe de 200 pra 210 -> cai a diferença na conta;
Dia 2: preço cai para 180 -> retira a diferença da sua conta.
Quando as perdas acumuladas ultrapassam a marge, você recebe da B3 uma chamada de margem (é o aviso que o corretor/bolsa faz quando sua conta não tem saldo suficiente para cobrir perdas em uma posição aberta).

OBS: a B3 estipula um valor de margem e de variação para cada produto. Exemplo: margem do contrato de mini dólar é 12% e sua variação máxima é 6% ao dia, ou seja, caso em algum dia o operador não possa pagar os 6%, é como se os 12% de margem já cobrissem essa falta. No fim, a B3 não fica com prejuízo e, como sanção, deixa de operar com essa contraparte. 


#### 2.3 Principais contratos

DI1: DI de 1 dia
Objetivo: Proteger ou apostar na taxa de juros (CDI) entre hoje e uma data futura. Quem compra DI1 está travando uma taxa de juros para aplicar dinheiro no futuro. Quem vende DI1 está apostando que os juros vão cair (ou precisa se proteger contra queda).
Exemplo:
Hoje o DI1 para jan/2026 está 13% ao ano.
Um banco que vai receber R$10 mi daqui 6 meses pode comprar DI1 para garantir que aplicará esse dinheiro a 13% ao ano, independente de onde o CDI estiver no futuro.


DOL: Dólar comercial
Objetivo: Proteger ou apostar na cotação do dólar comercial (PTAX). O preço do contrato reflete a expectativa do mercado para o dólar no vencimento. Quem compra DOL está comprado em dólar (aposta em alta do dólar). Quem vende DOL está vendido em dólar (aposta em queda).


IND: Ibovespa
Objetivo: Proteger ou apostar no índice de ações brasileiro (Ibovespa).
O preço do contrato segue a pontuação do Ibovespa no vencimento. Quem compra IND está comprado em ações (aposta em alta da bolsa). Quem vende IND está vendido em ações (aposta em queda). 

DDI: Cupom cambial sujo, contaminado pela variação cambial
Objetivo: Negociar a diferença entre juros no Brasil e nos EUA, mas de forma contaminada pela variação do dólar.
Fórmula mental: DDI = (taxa de juros brasileira) – (taxa de juros americana) + contaminação cambial
Como funciona na prática: O DDI é derivado da diferença entre o DI1 (juro real no Brasil) e o DOL (câmbio futuro). Ele mostra quanto o mercado espera de retorno acima do dólar.
Exemplo: Se o DI1 está 13% ao ano e o DOL está 5% acima do dólar spot, o DDI é aproximadamente 8% ao ano. Um investidor internacional compra DDI se acreditar que o juro real brasileiro vai subir mais do que o mercado espera, ou que o câmbio vai se desvalorizar menos que o previsto.
Por que "sujo" e "contaminado"? Porque ele não separa limpinho juro e câmbio. Uma variação no dólar altera o valor do DDI mesmo que os juros não tenham mudado. É um misturado de juro relativo + expectativa cambial.


#### 2.4 Cupom Cambial

Taxa de juros em dólares no Brasil: Remuneração no Brasil de depósitos locais em reais referenciados (valorizados) como se fossem dólares investidos localmente. Ou Seja, o quanto o investidor estrangeiro ganha pra investir seu dólar na taxa do Brasil sem a variação cambial.
Cupom cambial: diferença entre taxa de juros interna no Brasil e a variação cambial. Quanto maior o cupom, maior a atratividade para entrada de dólares.

Tipos de cupom cambial:
- DDI (cupom cambial sujo): é contaminado pela variação cambial, utiliza cotação da PTAX do dia anterior. Taxa spot até o vencimento.
- FRA (cupom cambial limpo): utiliza cotação do dólar spot  na data do vencimento. Taxa não é spot, e sim do primeiro vencimento até o segundo.

Cálculo da taxa de cupom cambial, exemplo:
Dólar spot: R$3
Dólar forward (60 dias úteis, 85 dias corridos): R$3,05
Taxa de juros doméstica (i): 10% a.a.

Taxa de cupom cambial = ((1 + i)^(du/252)) / ((dólar forward / dólar spot)) - 1 * (360/dc)


In [ ]:
# Dados para cálculo
dolar_spot = 3
dolar_forward = 3.05
i = 0.1
du = 60
dc = 85

# Cálculo da taxa de juros doméstica
taxa_juros_domestica = (1 + i) ** (du/252) - 1
print(f"Taxa de juros doméstica: {taxa_juros_domestica  :.2%} no período.")

# Cálculo da taxa de juros internacional
taxa_juros_internacional = (dolar_forward / dolar_spot) - 1
print(f"Taxa de juros internacional: {taxa_juros_internacional:.2%} no período.")

# Fazendo uma taxa pela outra e convertendo para a base 360
taxa_juros_domestica_base360 = (((1 + taxa_juros_domestica) / (1 + taxa_juros_internacional)) - 1) * (360/dc)
print(f"Taxa de juros doméstica convertida para base 360: {taxa_juros_domestica_base360:.2%} ao ano.")

# Cálculo direto
taxa_cupom_cambial = ((((1 + i) ** (du/252)) / (dolar_forward / dolar_spot)) - 1) * (360/dc)
print(f"Taxa de cupom cambial (cálculo direto): {taxa_cupom_cambial:.2%} ao ano.")

# Logo, a taxa de cupom cambial é de 2,62% ao ano, que é a diferença entre 2,30% doméstica efetiva e a variação cambial de 1,67% do dólar no período.

Taxa de juros doméstica: 2.30% no período.
Taxa de juros internacional: 1.67% no período.
Taxa de juros doméstica convertida para base 360: 2.62% ao ano.
Taxa de cupom cambial (cálculo direto): 2.62% ao ano.


#### 3. Contratos e Termo

O Contrato a termo representa um acordo entre as partes de compra e venda de uma certa quantidade de ativo (ou índice) em que as partes negociam o preço e a data futura para a liquidação da operação. Possui baixa liquidez e risco de crédito, uma vez que são contratos de balcão e não possui a B3 em nenhuma ponta (nesse caso, é somente REGISTRADO na B3), ficando com o risco de uma das partes não honrar com sua parte. De modo geral, o contrato a termo é mais flexível.

Principais diferenças entre contratos futuros e a termo:
- Futuro é padronizado e ajustado diariamente (com "dinheiro entrando e saindo da conta todo dia").
- Termo é customizado e o pagamento ocorre só no vencimento.
- Termo possui risco de crédito.



NDF (Non-deliverable forward): não há entrega do ativo ao final da operação, somente a exposição ao ativo.
Exemplo: Dólar R$5, uma importadora do ativo X faz um contrato de NDF de tamanho 50.000 com uma trava de R$5,10, com vencimento para 01/01/2027. Ao final da operação, caso o dólar esteja R$5,50, o contrato foi positivo para a importadora, da mesma forma, seria negativa se o dólar estivesse R$4,50 no final da operação.

#1Q: Ou seja, a outra parte para os 0,40 de diferença?

#2A: 

#2Q: o que acontece se uma das partes quiser sair da operação antes?

#2A:

Modalidades do termo de ações:
- Tradicional: o prazo é variável entre 16 e 999 dias; caso o ativo-objeto seja vendido antes do fim do prazo, o contrato é encerrado.
- Flexível: o ativo-objeto pode ser substituído, mas o prazo é variável entre 16 e 90 dias; o comprador pode vender os ativos e, com o dinheiro da venda (que pode ser usado comente pra aquisição de outro ativo), substituir o arquivo objeto sem que o contrato seja encerrado.


#### 3.1 Formação de preço do contrato a termo

VT: valor do contrato a termo
Vspot: valor à vista do ativo-objeto
T: tempo até o vencimento
R: taxa de mercado para o prazo T do contrato (no caso do contrato a termo de moedas, essa taxa é dada por um diferencial entre as taxas de captação e aplicação)

VT = Vspot (1 + R)^T

#### 4.1 Swap

Definição: contrato que estabelece a troca de fluxo de caixa e risco (indexador) entre duas contrapartes. Características:
- Contrato de obrigações recíprocas (o que ocorre é a troca de pagamentos)
- Valor inicial e fluxo de pagamentos são acordados entre as partes
- Liquidação feita no vencimento ou com ajuste periódico

In [17]:
# Exemplo de cálculo
NOTIONAL_BRL = 1_000_000
taxa_dolar_swap = 0.0086
CDI = 0.01
variacao_cambial = 0.005

ponta_a = NOTIONAL_BRL * (1 + CDI)
print(f"Ponta A (pagamento em dólar): R$ {ponta_a:,.2f}")

ponta_b = -NOTIONAL_BRL * (1 + taxa_dolar_swap) * (1 + variacao_cambial)
print(f"Ponta B (pagamento em real): R$ {ponta_b:,.2f}")

valor_pago_swap = ponta_a + ponta_b

print(f"Valor pago no swap: R$ {valor_pago_swap:,.2f}")


Ponta A (pagamento em dólar): R$ 1,010,000.00
Ponta B (pagamento em real): R$ -1,013,643.00
Valor pago no swap: R$ -3,643.00
